# MCQs: Trả lời câu hỏi trắc nghiệm

### Cấu hình

In [15]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))
from src.connection import get_connection

In [16]:
# Kiểm tra kết nối đến cơ sở dữ liệu
try:
    con = get_connection()
    print("Kết nối đến cơ sở dữ liệu thành công!")
except Exception as e:
    print(f"Kết nối đến cơ sở dữ liệu thất bại: {e}")

Kết nối đến cơ sở dữ liệu thành công!


## Q1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? 
(Tính từ orders.csv)

In [17]:
sql = """
WITH OrderGaps AS (
    -- Bước 1: Sắp xếp đơn hàng theo khách hàng và ngày, sau đó tính khoảng cách ngày
    SELECT 
        customer_id,
        order_date,
        LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) as previous_order_date,
        order_date - LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) as gap
    FROM orders
    WHERE order_status = 'delivered'
)
-- Bước 2: Tính trung vị của các khoảng cách (loại bỏ giá trị NULL của đơn hàng đầu tiên)
SELECT 
    percentile_cont(0.5) WITHIN GROUP (ORDER BY gap) as median_inter_order_gap
FROM OrderGaps
WHERE gap IS NOT NULL;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,median_inter_order_gap
0,175.0


Đáp án: C. 144
  
Giải thích: Đáp án gần nhất với 175

## Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

In [18]:
sql = """
SELECT 
    segment, 
    AVG((price - cogs) / price) AS avg_gross_margin
FROM products
GROUP BY segment
ORDER BY avg_gross_margin DESC;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,segment,avg_gross_margin
0,Standard,0.313442
1,Premium,0.285378
2,All-weather,0.284176
3,Activewear,0.265600
4,Performance,0.263650
5,Balanced,0.258038
6,Trendy,0.240759
7,Everyday,0.236343


Đáp án: D. Standard
  
Giải thích: Theo bảng trên

## Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [19]:
sql = """
SELECT 
    r.return_reason, 
    COUNT(*) AS reason_count
FROM returns r
JOIN products p ON r.product_id = p.product_id
WHERE p.category = 'Streetwear'
GROUP BY r.return_reason
ORDER BY reason_count DESC
LIMIT 1;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,return_reason,reason_count
0,wrong_size,7626


Đáp án: B. wrong_size
  
Giải thích:  wrong_size là cao nhất với 7626 lý do

## Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [20]:
sql = """
SELECT 
    traffic_source, 
    AVG(bounce_rate) AS avg_bounce_rate
FROM web_traffic
GROUP BY traffic_source
ORDER BY avg_bounce_rate ASC
LIMIT 1;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,traffic_source,avg_bounce_rate
0,email_campaign,0.004442


Đáp án: C. email_campaign
  
Giải thích: Theo bảng trên

## Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [21]:
sql = """
SELECT 
    (COUNT(promo_id) * 100.0 / COUNT(*)) AS promo_percentage
FROM order_items;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,promo_percentage
0,38.663493


Đáp án: C. 39%
  
Giải thích: Theo công thức đã cho thì đơn hàng do khuyến mãi chiếm xấp xỉ 39%

## Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [22]:
sql = """
SELECT 
    c.age_group, 
    COUNT(o.order_id) * 1.0 / COUNT(DISTINCT c.customer_id) AS avg_orders_per_customer
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE c.age_group IS NOT NULL
GROUP BY c.age_group
ORDER BY avg_orders_per_customer DESC;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,age_group,avg_orders_per_customer
0,55+,5.406851
1,45-54,5.357241
2,35-44,5.337343
3,25-34,5.245226
4,18-24,5.226656


Đáp án: A. 55+
  
Giải thích: Theo bảng trên

## Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

In [23]:
sql = """
SELECT 
    g.region, 
    SUM(oi.quantity * oi.unit_price) AS total_revenue
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN geography g ON o.zip = g.zip
GROUP BY g.region
ORDER BY total_revenue DESC;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,region,total_revenue
0,East,7.637533e+09
1,Central,4.941908e+09
2,West,3.851035e+09


Đáp án: C. East
  
Giải thích: Theo bảng trên

## Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [24]:
sql = """
SELECT 
    payment_method, 
    COUNT(*) AS total_cancelled
FROM orders
WHERE order_status = 'cancelled'
GROUP BY payment_method
ORDER BY total_cancelled DESC
LIMIT 1;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,payment_method,total_cancelled
0,credit_card,28452


Đáp án: A. credit_card
  
Giải thích: Theo bảng trên

## Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

In [25]:
sql = """
SELECT 
    p.size, 
    COUNT(r.return_id) * 1.0 / COUNT(oi.*) AS return_rate
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN returns r ON oi.order_id = r.order_id AND oi.product_id = r.product_id
WHERE p.size IN ('S', 'M', 'L', 'XL')
GROUP BY p.size
ORDER BY return_rate DESC;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,size,return_rate
0,S,0.056515
1,L,0.056250
2,M,0.055682
3,XL,0.055200


Đáp án: A. S
  
Giải thích: Theo bảng trên

## Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [26]:
sql = """
SELECT 
    installments, 
    AVG(payment_value) AS avg_payment_amount
FROM payments
GROUP BY installments
ORDER BY avg_payment_amount DESC
LIMIT 1;
"""
df = con.execute(sql).df()
# Hiển thị kết quả
df

,installments,avg_payment_amount
0,6,24446.654403


Đáp án: C. 6 kỳ
  
Giải thích: Theo bảng trên

## PHÂN TÍCH CHUYÊN SÂU HÀNH VI KHÁCH HÀNG & DANH MỤC
Dựa trên kết quả truy vấn từ hệ thống cơ sở dữ liệu, chúng tôi xác lập 4 trụ cột phân tích trọng tâm để giải quyết bài toán tối ưu hóa lợi nhuận.

1. Định lượng "Cơn nghiện Khuyến mãi" (Evidence of Promotion Dependency)

- Phân tích tỷ lệ áp dụng mã giảm giá trên mỗi dòng sản phẩm bán ra cho thấy mức độ lệ thuộc cực lớn của doanh nghiệp vào các chương trình ưu đãi:

- Chỉ số định lượng: Khoảng 38,66% tổng số mặt hàng trong danh mục order_items được bán ra có kèm theo mã khuyến mãi (promo_id không null).

- Phân tích BA: Cứ trung bình 3 món hàng được bán thì có hơn 1 món hàng cần "mồi" giảm giá để kích hoạt hành vi mua.

- Hệ lụy hệ thống: Điều này xác nhận doanh số hiện tại không hoàn toàn đến từ giá trị cốt lõi của sản phẩm mà bị chi phối bởi áp lực giá. Việc cắt giảm ngân sách khuyến mãi đột ngột có thể dẫn đến sự sụt giảm quy mô (Volume) nghiêm trọng nếu không có chiến lược chuyển đổi thông minh.

2. Xác định "Vùng an toàn" cho Biên lợi nhuận (Margin Safe Zones)

- Để cứu vãn tình trạng Margin âm, doanh nghiệp cần tập trung nguồn lực vào các nhóm sản phẩm có "sức khỏe" tài chính tốt nhất:

- Phân khúc tiềm năng: Phân khúc Standard (Tiêu chuẩn) đang dẫn đầu về hiệu quả tài chính với tỷ suất lợi nhuận gộp trung bình đạt xấp xỉ 31,34%.

- So sánh đối chiếu: Tỷ suất này cao vượt trội so với các nhóm khác như Everyday (23,63%) hay Trendy (24,07%).

- Đề xuất chiến lược cho AI: Khi triển khai mô hình LightGBM, cần thiết lập trọng số ưu tiên bảo vệ Margin cho nhóm Standard. Đây là nhóm sản phẩm "nồi cơm" (Cash Cow), nơi doanh nghiệp có thể duy trì mức giá ổn định mà không cần giảm giá quá sâu vẫn đảm bảo được lợi nhuận gộp.

3. Chân dung Khách hàng & Chu kỳ Mua sắm (Customer Persona & Lifecycle)

- Việc hiểu rõ "điểm rơi" mua sắm của khách hàng là chìa khóa để phân bổ khuyến mãi cá nhân hóa:

- Nhóm khách hàng trung thành: Khách hàng thuộc độ tuổi 55+ có tần suất mua hàng cao nhất với trung bình 5,41 đơn hàng/khách hàng.

- Chu kỳ tái mua hàng (Inter-order gap): Đối với nhóm khách hàng phát sinh nhiều hơn một đơn hàng, trung vị khoảng cách giữa hai lần mua liên tiếp là 175 ngày (tương đương khoảng 6 tháng).

- Chiến thuật tiếp cận:

    - AI có thể sử dụng mốc 175 ngày làm ngưỡng cảnh báo rời bỏ (Churn warning).

    - Ví dụ: Nếu một khách hàng thuộc nhóm 55+ đã quá 5 tháng chưa phát sinh đơn hàng mới, đây là "thời điểm vàng" để gửi thông điệp chăm sóc hoặc khuyến mãi nhẹ để kéo họ quay lại ngay trước khi chu kỳ 6 tháng kết thúc.

4. Tối ưu hóa Kênh Chuyển đổi (Conversion Channel Optimization)

- Dữ liệu lưu lượng truy cập Web cung cấp manh mối về nơi cần tập trung ngân sách Marketing để đạt hiệu quả cao nhất:

    - Kênh chất lượng nhất: Nguồn Email Campaign ghi nhận tỷ lệ thoát (Bounce rate) thấp nhất, trung bình chỉ ở mức 0,44%.

    - Đánh giá chất lượng: Tỷ lệ thoát cực thấp cho thấy khách hàng đến từ Email có ý định mua sắm rất rõ ràng và mức độ quan tâm sâu đối với nội dung thông điệp.

    - Giải pháp tối ưu: Thay vì triển khai giảm giá hàng loạt trên Website (gây lãng phí Margin cho cả nhóm khách vãng lai), AI nên ưu tiên chiến lược "Khuyến mãi ẩn" thông qua Email cho từng cá nhân để tối đa hóa tỷ lệ chuyển đổi trên mỗi đồng vốn bỏ ra.